In [4]:
pip install opencv-python numpy pandas matplotlib pillow pytesseract

Note: you may need to restart the kernel to use updated packages.


In [8]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pytesseract
import re
from datetime import datetime, timedelta
from pathlib import Path
from PIL import Image

# =========================
# 0. TESSERACT 경로 설정 (Windows)
# =========================
# C:\Program Files\Tesseract-OCR 경로 확인했다고 했으니 그대로 사용
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"


# =========================
# 1. 상단 기간 텍스트 OCR
# =========================
def extract_date_range_from_header(img_rgb):
    """
    상단 '时间范围 2023-08-27 ~ 2025-11-26' 부분을 OCR로 읽어
    시작일/종료일(datetime)을 반환.
    이 비율 설정은 961x535 짜리 바이두 지수 캡쳐 템플릿 기준임.
    """
    h, w, _ = img_rgb.shape

    # 제목 아래 ~ 필터 영역 아래까지 대략 포함
    top = int(h * 0.12)
    bottom = int(h * 0.26)
    left = int(w * 0.30)
    right = int(w * 0.85)

    header = img_rgb[top:bottom, left:right]
    gray = cv2.cvtColor(header, cv2.COLOR_RGB2GRAY)
    gray = cv2.threshold(gray, 0, 255,
                         cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]

    text = pytesseract.image_to_string(
        gray, lang="chi_sim+eng", config="--psm 6"
    )

    # yyyy-mm-dd 패턴 2개 이상 추출
    dates = re.findall(r"\d{4}-\d{2}-\d{2}", text)
    if len(dates) < 2:
        raise ValueError(f"[DATE OCR FAIL] 기간 텍스트 인식 실패: {text!r}")

    start_str, end_str = dates[0], dates[1]
    start_date = datetime.strptime(start_str, "%Y-%m-%d")
    end_date = datetime.strptime(end_str, "%Y-%m-%d")
    return start_date, end_date, text


# =========================
# 2. 왼쪽 Y축 숫자 OCR
# =========================
def extract_y_axis_range(img_rgb):
    """
    왼쪽 Y축 눈금 숫자들을 OCR로 읽어 최솟값/최댓값을 반환.
    예) 30~210, 100~7000 등.
    """
    h, w, _ = img_rgb.shape

    left = int(w * 0.05)
    right = int(w * 0.16)
    top = int(h * 0.28)
    bottom = int(h * 0.90)

    y_axis = img_rgb[top:bottom, left:right]
    gray = cv2.cvtColor(y_axis, cv2.COLOR_RGB2GRAY)
    gray = cv2.GaussianBlur(gray, (3, 3), 0)
    gray = cv2.threshold(gray, 0, 255,
                         cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]

    text = pytesseract.image_to_string(
        gray, lang="eng", config="--psm 6 digits"
    )

    nums = re.findall(r"\d+", text)
    nums = list(map(int, nums)) if nums else []

    if len(nums) < 2:
        # OCR이 잘 안되는 경우 기본값으로 세팅 (필요하면 여기 값 조정 가능)
        print(f"[WARN] Y축 OCR 실패/부족, text={text!r} → 임시 30~210 사용")
        return 30, 210, text

    y_min = min(nums)
    y_max = max(nums)
    return y_min, y_max, text


# =========================
# 3. 그래프 ROI(곡선이 그려진 영역) 자동 탐지
# =========================
def detect_graph_roi(img_rgb):
    """
    그래프가 있는 대략 위치(중앙~하단)에서
    하늘색 라인 마스크를 이용해 실제 곡선이 있는 bounding box를 찾음.
    """
    h, w, _ = img_rgb.shape

    # 그래프가 있을 법한 대략 영역
    rough_top = int(h * 0.27)
    rough_bottom = int(h * 0.89)
    rough_left = int(w * 0.12)
    rough_right = int(w * 0.97)

    roi = img_rgb[rough_top:rough_bottom, rough_left:rough_right]
    roi_bgr = cv2.cvtColor(roi, cv2.COLOR_RGB2BGR)
    hsv = cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2HSV)

    # 바이두 그래프 라인(하늘색 계열) HSV 범위
    lower_cyan = np.array([80, 40, 40])
    upper_cyan = np.array([130, 255, 255])
    mask = cv2.inRange(hsv, lower_cyan, upper_cyan)

    # 끊긴 부분 보정
    kernel = np.ones((3, 3), np.uint8)
    mask = cv2.dilate(mask, kernel, iterations=1)

    contours, _ = cv2.findContours(
        mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    if not contours:
        print("[WARN] ROI 탐지 실패 → rough 영역 전체 사용")
        return rough_left, rough_top, rough_right, rough_bottom

    cnt = max(contours, key=cv2.contourArea)
    x, y, w_roi, h_roi = cv2.boundingRect(cnt)

    x1 = rough_left + x
    y1 = rough_top + y
    x2 = rough_left + x + w_roi
    y2 = rough_top + y + h_roi

    margin = 5
    x1 = max(0, x1 - margin)
    y1 = max(0, y1 - margin)
    x2 = min(w, x2 + margin)
    y2 = min(h, y2 + margin)

    return x1, y1, x2, y2


# =========================
# 4. ROI 안에서 곡선 픽셀 → 지수값으로 변환
# =========================
def extract_curve_from_roi(img_rgb, x1, y1, x2, y2, y_min_value, y_max_value):
    roi = img_rgb[y1:y2, x1:x2]
    roi_bgr = cv2.cvtColor(roi, cv2.COLOR_RGB2BGR)
    hsv = cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2HSV)

    lower_cyan = np.array([80, 40, 40])
    upper_cyan = np.array([130, 255, 255])
    mask = cv2.inRange(hsv, lower_cyan, upper_cyan)

    h_roi, w_roi = mask.shape

    x_pixels = []
    values = []

    for x in range(w_roi):
        ys = np.where(mask[:, x] > 0)[0]
        if len(ys) == 0:
            x_pixels.append(x)
            values.append(np.nan)
        else:
            # 여러 픽셀이 있으면 중앙값 사용
            y_med = int(np.median(ys))
            # 0(위) ~ h_roi-1(아래) → y_max_value ~ y_min_value로 선형 변환
            ratio = y_med / (h_roi - 1)
            v = y_max_value - ratio * (y_max_value - y_min_value)
            x_pixels.append(x)
            values.append(v)

    return np.array(x_pixels), np.array(values)


# =========================
# 5. x픽셀 → 날짜로 매핑
# =========================
def map_pixels_to_dates(x_pixels, start_date, end_date):
    x_min = x_pixels.min()
    x_max = x_pixels.max()
    w = x_max - x_min
    if w <= 0:
        raise ValueError("ROI width가 0 이하")

    total_days = (end_date - start_date).days
    dates = []
    for x in x_pixels:
        ratio = (x - x_min) / w
        d = start_date + timedelta(days=ratio * total_days)
        dates.append(d)
    return np.array(dates)


# =========================
# 6. 한 장 처리 메인 함수
# =========================
def process_baidu_index_image(img_path: str, out_prefix: str = None,
                              show_plot: bool = True):
    """
    PNG 바이두 지수 스크린샷 한 장을 처리:
      - 기간 OCR
      - Y축 범위 OCR
      - 그래프 ROI 자동 탐지
      - 곡선 복원 (픽셀 → 지수)
      - raw / 일별 집계 엑셀 저장
    """
    img_path = Path(img_path)
    if out_prefix is None:
        out_prefix = img_path.stem

    # Pillow로 읽어서 한글/공백 경로 문제 해결
    try:
        img_pil = Image.open(str(img_path)).convert("RGB")
    except Exception as e:
        raise FileNotFoundError(f"이미지 열기 실패: {img_path} / {e}")

    img_rgb = np.array(img_pil)

    # 1) 기간 OCR
    start_date, end_date, header_text = extract_date_range_from_header(img_rgb)
    print(f"\n[INFO] {img_path.name}")
    print(f"  기간: {start_date.date()} ~ {end_date.date()}")
    print(f"  header OCR: {header_text!r}")

    # 2) Y축 OCR
    y_min, y_max, y_text = extract_y_axis_range(img_rgb)
    print(f"  Y축 범위: {y_min} ~ {y_max}")
    print(f"  y-axis OCR: {y_text!r}")

    # 3) 그래프 ROI 탐지
    x1, y1, x2, y2 = detect_graph_roi(img_rgb)
    print(f"  ROI: x1={x1}, y1={y1}, x2={x2}, y2={y2}")

    # 4) 곡선 복원
    x_pixels, values = extract_curve_from_roi(
        img_rgb, x1, y1, x2, y2, y_min, y_max
    )
    dates = map_pixels_to_dates(x_pixels, start_date, end_date)

    df_raw = pd.DataFrame({
        "pixel_x": x_pixels,
        "date": dates,
        "value": values
    })

    # 5) 일별 집계 (최고/최저/평균)
    df_raw["date_day"] = df_raw["date"].dt.date
    daily = df_raw.groupby("date_day")["value"].agg(
        ["min", "max", "mean"]
    ).reset_index()
    daily = daily.rename(
        columns={"min": "daily_min",
                 "max": "daily_max",
                 "mean": "daily_mean"}
    )

    # 6) 엑셀 저장
    raw_path = img_path.with_name(f"{out_prefix}_raw_pixel_series.xlsx")
    daily_path = img_path.with_name(f"{out_prefix}_daily_series.xlsx")

    df_raw.to_excel(raw_path, index=False)
    daily.to_excel(daily_path, index=False)

    print(f"  [SAVE] raw   → {raw_path.name}")
    print(f"  [SAVE] daily → {daily_path.name}")

    # 7) 복원 곡선 확인용 그래프
    if show_plot:
        plt.figure(figsize=(12, 4))
        plt.plot(df_raw["date"], df_raw["value"], linewidth=1)
        plt.title(f"{out_prefix} - 복원 곡선")
        plt.xlabel("date")
        plt.ylabel("index value")
        plt.grid(True)
        plt.tight_layout()
        plt.show()

    return df_raw, daily


# =========================
# 7. 폴더 안의 PNG 여러 장 한 번에 처리
# =========================
if __name__ == "__main__":
    image_dir = Path(".")   # 이 스크립트가 있는 폴더
    for png in image_dir.glob("*.png"):
        try:
            process_baidu_index_image(str(png), show_plot=False)
        except Exception as e:
            print(f"[ERROR] {png.name}: {e}")



[INFO] 건성 피부  건조 피부.png
  기간: 2023-08-27 ~ 2025-11-26
  header OCR: ') YTB) 2023-08-27 ~ 2025-11-26\n'
  Y축 범위: 1 ~ 20231113
  y-axis OCR: '1\n20231113\n'
  ROI: x1=110, y1=193, x2=937, y2=365
  [SAVE] raw   → 건성 피부  건조 피부_raw_pixel_series.xlsx
  [SAVE] daily → 건성 피부  건조 피부_daily_series.xlsx

[INFO] 광대 넓음.png
  기간: 2023-09-04 ~ 2025-11-26
  header OCR: ') YTB) 2023-09-04 ~ 2025-11-26\n'
  Y축 범위: 1 ~ 20231127
  y-axis OCR: '1\n\n1\n\n20231127\n6\n'
  ROI: x1=110, y1=239, x2=937, y2=335
  [SAVE] raw   → 광대 넓음_raw_pixel_series.xlsx
  [SAVE] daily → 광대 넓음_daily_series.xlsx

[INFO] 광대 돌출.png
  기간: 2023-09-04 ~ 2025-11-26
  header OCR: ') YTB) 2023-09-04 ~ 2025-11-26\n'
  Y축 범위: 11 ~ 2023
  y-axis OCR: '2023-11-27\n'
  ROI: x1=110, y1=216, x2=937, y2=302
  [SAVE] raw   → 광대 돌출_raw_pixel_series.xlsx
  [SAVE] daily → 광대 돌출_daily_series.xlsx

[INFO] 기미  멜라스마.png
  기간: 2023-08-27 ~ 2025-11-26
  header OCR: ') YTB) 2023-08-27 ~ 2025-11-26\n'
  Y축 범위: 11 ~ 2023
  y-axis OCR: '100\n2023-11-13\n-.\

In [9]:
import pytesseract
print(pytesseract.get_tesseract_version())

5.5.0.20241111
